In [40]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from typing import TypedDict



In [41]:
load_dotenv()
model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

In [42]:
class quadState(TypedDict):
    a : str
    b : str
    c : str
    eqn : str
    discriminant : float
    result : str


In [43]:
def show_eqn(state : quadState) -> str:
    equation = f"{state['a']}x^2 + {state['b']}x + {state['c']} = 0"
    return {'eqn' : equation}


In [44]:
def calculate_discriminant(state : quadState) -> float:
    a = float(state['a'])
    b = float(state['b'])
    c = float(state['c'])
    discriminant = b**2 - 4*a*c
    return {'discriminant' : discriminant}

In [45]:
def realroot(state : quadState) -> str:
    root1 = -state['b'] + (state['discriminant']**0.5) / (2*float(state['a']))
    root2 = -state['b'] - (state['discriminant']**0.5) / (2*float(state['a']))
    return {'result' : f"The roots are real and distinct: {root1} and {root2}"}

In [46]:
def singleroot(state : quadState) -> str:
    root = -state['b'] / (2*float(state['a']))
    return {'result' : f"The roots are real and equal: {root}"}

In [47]:
def complexroot(state : quadState) -> str:
    return {'result' : "The roots are complex and distinct."}

In [48]:
def condition(state : quadState) -> str:
    if state['discriminant'] > 0:
        return "realroots"
    elif state['discriminant'] == 0:
        return "singleroot"
    else:
        return "complexroots"

In [49]:
graph = StateGraph(quadState)
graph.add_node("show_eqn", show_eqn)
graph.add_node("calculate_discriminant", calculate_discriminant)
graph.add_node("realroots", realroot)
graph.add_node("complexroots", complexroot)
graph.add_node("singleroot", singleroot)
graph.add_edge(START, "show_eqn")
graph.add_edge("show_eqn", "calculate_discriminant")
graph.add_conditional_edges("calculate_discriminant", condition)
graph.add_edge("realroots", END)
graph.add_edge("complexroots", END)
graph.add_edge("singleroot", END)
workflow = graph.compile()


In [52]:
initial_state = {
    "a": 1,
    "b": -1,
    "c": 4
}
workflow.invoke(initial_state)

{'a': 1,
 'b': -1,
 'c': 4,
 'eqn': '1x^2 + -1x + 4 = 0',
 'discriminant': -15.0,
 'result': 'The roots are complex and distinct.'}